In [ ]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import numpy as np
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.linalg import eigh
import sympy
import openfermion as op
import pyvista as pv
import torch
import cirq

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import warnings
!pip install biopython
from Bio.PDB import PDBParser

In [4]:
protein = 'proteins/1ENH.pdb'
tensor = pdb_voxelizier.pdb_to_tensor(protein, grid_size=32)
model = cnn_mlp_encoder.ProteinPhysicsEncoder(num_sites=4)

Get feature maps for first convolutional layer 

In [22]:
def visualize_feature_maps(feature_maps, slice_index=None):
    #so gradients are not calculated when running 
    # Remove batch dimension and move to CPU
    maps = feature_maps.squeeze(0).cpu()
    # calculates subplot rows 
    # len: gives how many items are in a tensor of feature maps 
    rows = (len(maps) + 3) // 4
    # creates figure; widthxheightxrows, 16x4 just general choice 
    plt.figure(figsize=(12, 3 * rows), dpi = 200)
    # loop that goes through every feature map 
    #plt.subplots_adjust(wspace=0, hspace=0)
    # enumerate: adds an index so converts list into index
    for i, fmap in enumerate(maps):
        # chooses the slice
        z = slice_index if slice_index is not None else fmap.shape[0] // 2
        # gets the 2d slice 
        slice_img = fmap[z].numpy()
        slice_img = np.rot90(slice_img)
        # normalizes the values 
        slice_img = (slice_img - slice_img.min()) / (slice_img.max() - slice_img.min() + 1e-8)
        # create subplot and display 
        plt.subplot(rows, 4, i + 1)
        plt.imshow(slice_img, cmap="viridis", interpolation="nearest")
        plt.title(f"Map {i+1}", fontsize = 18)
        plt.axis("off")

    #plt.tight_layout()
    plt.show()


In [ ]:
# converts data into pytorch tensor 
input_tensor = torch.FloatTensor(tensor)
# try changing slice to see different sections of 3d feature maps  
# this is the forward pass function as need to apply first convolutional layer and the operations inorder
with torch.no_grad():
    # First convolutional layer
    conv1_maps = F.relu(model.conv1(input_tensor))
    # Pooling
    pooled = model.pool1(conv1_maps)
    # Second convolutional layer
    conv2_maps = F.relu(model.conv2(pooled))

In [ ]:
#for z in [6, 8, 10, 12]:
maps = conv1_maps.squeeze(0)

slice_scores = maps.abs().sum(dim=(0,2,3))
best_slice = torch.argmax(slice_scores).item()

print(best_slice)

visualize_feature_maps(conv1_maps, slice_index=best_slice)

In [ ]:
visualize_feature_maps(conv1_maps, slice_index=13)

get feature maps for second convolutional layer 

In [ ]:

# try changing slice to see different sections of 3d feature maps  
#for z in [2, 4, 6, 8, 10, 12, 14]:
maps = conv2_maps.squeeze(0)

slice_scores = maps.abs().sum(dim=(0,2,3))
best_slice = torch.argmax(slice_scores).item()

print(best_slice)

visualize_feature_maps(conv2_maps, slice_index=best_slice)
#visualize_feature_maps(conv2_maps, slice_index=8)

In [ ]:
for z in [4, 8, 12]:
    print(f"Slice {z}")
    visualize_feature_maps(conv2_maps, slice_index=z)

In [22]:
def visualize_filters(layer):
# get learned filter weights 
# detach = removes weights from computation graph, so it doesn't compute gradients only weights
  filters = layer.weight.detach().cpu()
# shape (out channels, in channels, depth, height, width), so only takes out channels needed
  num_filters = filters.shape[0]
# creates figure, just general size 12x8
  plt.figure(figsize = (12, 8))
# loop to go through every filter 
  for i in range(num_filters): 
    # get one filter, so first input channel 
    filter = filters[i, 0]
    # show middle slide of 3d filter, 
    middle_slice = filter[1]
    # create grid of plots 
    plt.subplot(4,4, i + 1)
    # convert numbers to pixels and use grayscale so positive weights = lighter vs negative = darker
    plt.imshow(middle_slice, cmap = "viridis")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.show()


In [ ]:
visualize_filters(model.conv1)


In [ ]:
visualize_filters(model.conv2)


In [ ]:
with torch.no_grad():
    x = model.pool1(F.relu(model.conv1(input_tensor)))
    x = model.pool2(F.relu(model.conv2(x)))
    x = x.view(x.size(0), -1)
    embedding = F.relu(model.fc1(x))

print(embedding.shape)

In [ ]:
plt.figure(figsize=(14, 4))

plt.bar(range(128), embedding.numpy().flatten())

plt.xlabel("Embedding Dimension")
plt.ylabel("Activation")
plt.title("128-D Latent Embedding")

plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

def visualize_filter_voxels(conv_layer, filter_num=0):

    kernel = conv_layer.weight.detach().cpu()[filter_num,0]

    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')

    ax.voxels(kernel.numpy() > 0)

    ax.set_title(f'Filter {filter_num}')
    plt.show()

In [ ]:
visualize_filter_voxels(model.conv1, filter_num=1)